In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

print("1. Kendi verisetiniz (CSV) yükleniyor...")
# Veriyi doğrudan bilgisayarınızdaki CSV dosyasından yüklüyoruz
veri = pd.read_csv("groq_sentetik_10000_ETIKETLI.csv")

# Boş (NaN) satırlar varsa hatayı önlemek için temizliyoruz
# Sadece orijinal ve etiket sütunuyla çalışacağımız için bu ikisini kontrol etmemiz yeterli
veri = veri.dropna(subset=['orijinal', 'label'])

print(f"Toplam {len(veri)} adet veri başarıyla yüklendi.")

print("\n2. Orijinal veriler ilk %80 eğitim, son %20 test olacak şekilde sıralı ayrılıyor...")
# İşin sırrı burada: Doğrudan orijinal metinleri bölüyoruz.
# shuffle=False parametresi verinin rastgele karıştırılmasını engeller.
X_train_metin, X_test_metin, y_train, y_test = train_test_split(
    veri['orijinal'], 
    veri['label'], 
    test_size=0.2, 
    shuffle=False
)

print("3. Metinler sayısallaştırılıyor (TF-IDF)...")
# Kelime dağarcığını oluşturuyoruz
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# Modeli eğitim setindeki (ilk %80 orijinal) kelimelere göre eğitiyoruz (fit) ve sayısallaştırıyoruz (transform)
X_train = vectorizer.fit_transform(X_train_metin)

# Test edeceğimiz metinleri (son %20 orijinal), öğrendiğimiz kelime dağarcığına göre sayısallaştırıyoruz (Sadece transform)
X_test = vectorizer.transform(X_test_metin)

print("4. Model eğitiliyor (Lojistik Regresyon)...")
# Lojistik Regresyon modelini kuruyor ve eğitiyoruz
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("5. Tahminler yapılıyor ve sonuçlar hesaplanıyor...\n")
sonuc = model.predict(X_test)

# --- DEĞERLENDİRME VE RAPORLAMA ---
print("-" * 50)
print("SONUÇLAR VE METRİKLER (Lojistik Regresyon - Orijinal Eğitim / Orijinal Test)")
print("-" * 50)

# Confusion Matrix
cm = confusion_matrix(y_test, sonuc)
print("Karmaşıklık Matrisi (Confusion Matrix):\n", cm)

# Doğruluk
dogruluk = accuracy_score(y_test, sonuc)
print(f"\nDoğruluk Değeri (Accuracy): %{dogruluk * 100:.2f}")

# Diğer Metrikler
kesinlik = precision_score(y_test, sonuc)
duyarlilik = recall_score(y_test, sonuc)
f1 = f1_score(y_test, sonuc)

print(f"Kesinlik (Precision): {kesinlik:.4f}")
print(f"Duyarlılık (Recall): {duyarlilik:.4f}")
print(f"F1 Skoru: {f1:.4f}\n")

print("Sınıflandırma Raporu (Classification Report):")
print(classification_report(y_test, sonuc, target_names=['Sınıf 0', 'Sınıf 1']))

1. Kendi verisetiniz (CSV) yükleniyor...
Toplam 10000 adet veri başarıyla yüklendi.

2. Orijinal veriler ilk %80 eğitim, son %20 test olacak şekilde sıralı ayrılıyor...
3. Metinler sayısallaştırılıyor (TF-IDF)...
4. Model eğitiliyor (Lojistik Regresyon)...
5. Tahminler yapılıyor ve sonuçlar hesaplanıyor...

--------------------------------------------------
SONUÇLAR VE METRİKLER (Lojistik Regresyon - Orijinal Eğitim / Orijinal Test)
--------------------------------------------------
Karmaşıklık Matrisi (Confusion Matrix):
 [[831 180]
 [120 869]]

Doğruluk Değeri (Accuracy): %85.00
Kesinlik (Precision): 0.8284
Duyarlılık (Recall): 0.8787
F1 Skoru: 0.8528

Sınıflandırma Raporu (Classification Report):
              precision    recall  f1-score   support

     Sınıf 0       0.87      0.82      0.85      1011
     Sınıf 1       0.83      0.88      0.85       989

    accuracy                           0.85      2000
   macro avg       0.85      0.85      0.85      2000
weighted avg       